# 02 — Cloud Cluster Simulator

Custom OpenAI Gymnasium environment for the RL cloud resource manager.
Generates realistic workloads from Google Cluster Trace 2011 patterns
(loaded from `trace_params.json`), models CPU **and** memory, tracks SLA
deadlines, and produces a 32-dim interval-robust state vector.

Run top-to-bottom.

## 1. Setup and constants

In [1]:
import json
import numpy as np
import gymnasium as gym
from gymnasium import spaces

# load the trace patterns extracted in notebook 01
with open('trace_params.json') as f:
    trace_params = json.load(f)
stats = trace_params['stats']            # stats[day][hour] -> arrival_rate, avg_cpu, ...

# ---- cluster configuration ----
MIN_PODS = 2
MAX_PODS = 20
VM_CPU_CAP = 1.0                          # CPU capacity per VM
VM_MEM_CAP = 1.0                          # memory capacity per VM
JOBS_PER_STEP_PER_VM = 25                 # nominal throughput (for utilisation/state)

# ---- timing ----
STEP_MINUTES = 15
STEPS_PER_HOUR = 60 // STEP_MINUTES       # 4
STEPS_PER_DAY  = STEPS_PER_HOUR * 24      # 96
STEPS_PER_WEEK = STEPS_PER_DAY * 7        # 672  (one episode = one week)

# ---- workload scaling (calibrated so 2-20 VM range is meaningful) ----
WORKLOAD_SCALE = 0.008

# ---- reward weights (balanced config; varied later for Pareto frontier) ----
LAMBDA_COST = 0.4
LAMBDA_SLA  = 1.0
LAMBDA_UTIL = 0.2

print("Setup complete. Steps per week:", STEPS_PER_WEEK)

Setup complete. Steps per week: 672


## 2. The environment

The full `CloudClusterEnv` in one place — CPU and memory, job generation,
VM packing (a job needs both CPU and memory room), SLA deadlines, reward,
32-dim state, decision log, and hint slots.

In [4]:
class CloudClusterEnv(gym.Env):
    """Simulated cloud cluster. The agent chooses how many VMs to run,
    balancing cost against SLA compliance, using CPU and memory signals."""

    def __init__(self):
        super().__init__()
        # continuous action: one value in [-1, 1] -> scaling delta
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        # 32-dim observation (29 core + 3 hint slots)
        self.observation_space = spaces.Box(0.0, 1.0, shape=(32,), dtype=np.float32)

    # ---------------------------------------------------------------- reset
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.active_vms = 4
        self.queue = []
        self.cost_total = 0.0
        self.total_breaches = 0
        self.history = [[0.0, 0.0, 0.0, 0.0] for _ in range(5)]  # avg/max cpu, avg/max mem
        self.prev_queue_len = 0
        # hint slots (0 until synthetic-hint training / operator input)
        self.hint_active = 0.0
        self.hint_magnitude = 0.0
        self.hint_time_to_event = 0.0
        # decision log -> feeds the Phase 2 RAG explainability layer
        self.decision_log = []
        return self._build_state(), {}

    # --------------------------------------------------------- time helper
    def _current_day_hour(self):
        total_hours = self.step_count // STEPS_PER_HOUR
        hour = int(total_hours % 24)
        day = int((total_hours // 24) % 7)
        return day, hour

    # ------------------------------------------------------ job generation
    def _generate_jobs(self):
        day, hour = self._current_day_hour()
        s = stats[str(day)][str(hour)]
        jobs_this_step_mean = (s['arrival_rate'] * WORKLOAD_SCALE) / STEPS_PER_HOUR
        n_jobs = np.random.poisson(jobs_this_step_mean)
        deadline_by_class = {0: 2, 1: 4, 2: 8, 3: 16}
        new_jobs = []
        for _ in range(n_jobs):
            cpu = float(np.clip(np.random.normal(s['avg_cpu'], s['cpu_std']), 0.001, 1.0))
            mem = float(np.clip(np.random.normal(s['avg_mem'], s['mem_std']), 0.001, 1.0))
            sched = int(np.random.choice([0, 1, 2, 3], p=s['class_distribution']))
            new_jobs.append({'cpu': cpu, 'mem': mem,
                             'deadline': self.step_count + deadline_by_class[sched]})
        return new_jobs

    # ------------------------------------------------- assign jobs to VMs
    def _assign_jobs(self):
        vm_cpu_load = [0.0] * self.active_vms
        vm_mem_load = [0.0] * self.active_vms
        still_waiting = []
        jobs_processed = 0
        for job in self.queue:
            placed = False
            for vm_idx in sorted(range(self.active_vms), key=lambda i: vm_cpu_load[i]):
                cpu_ok = vm_cpu_load[vm_idx] + job['cpu'] <= VM_CPU_CAP
                mem_ok = vm_mem_load[vm_idx] + job['mem'] <= VM_MEM_CAP
                if cpu_ok and mem_ok:          # needs BOTH cpu and mem room
                    vm_cpu_load[vm_idx] += job['cpu']
                    vm_mem_load[vm_idx] += job['mem']
                    jobs_processed += 1
                    placed = True
                    break
            if not placed:
                still_waiting.append(job)
        self.queue = still_waiting
        if self.active_vms > 0:
            avg_cpu = float(np.mean(vm_cpu_load)); max_cpu = float(np.max(vm_cpu_load))
            avg_mem = float(np.mean(vm_mem_load)); max_mem = float(np.max(vm_mem_load))
        else:
            avg_cpu = max_cpu = avg_mem = max_mem = 0.0
        return jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem

    # ---------------------------------------------------- deadline checks
    def _check_deadlines(self):
        breaches = 0
        surviving = []
        for job in self.queue:
            if self.step_count > job['deadline']:
                breaches += 1
            else:
                surviving.append(job)
        self.queue = surviving
        return breaches

    # ---------------------------------------------------------- the state
    def _build_state(self):
        hist = np.array(self.history, dtype=np.float32)
        avg_cpu_hist = hist[:, 0]; max_cpu_hist = hist[:, 1]
        avg_mem_hist = hist[:, 2]; max_mem_hist = hist[:, 3]

        capacity = self.active_vms * JOBS_PER_STEP_PER_VM
        queue_depth_ratio = float(np.clip(
            len(self.queue) / capacity if capacity > 0 else 1.0, 0.0, 1.0))

        growth = len(self.queue) - self.prev_queue_len
        queue_growth = float(np.clip(
            0.5 + (growth / capacity if capacity > 0 else 0.0), 0.0, 1.0))

        if len(self.queue) > 0:
            near = sum(1 for j in self.queue if j['deadline'] - self.step_count <= 2)
            sla_pressure = near / len(self.queue)
            slacks = [j['deadline'] - self.step_count for j in self.queue]
            min_slack = float(np.clip(min(slacks) / 16.0, 0.0, 1.0))
        else:
            sla_pressure = 0.0
            min_slack = 1.0

        active_vms_norm = self.active_vms / MAX_PODS
        cost_norm = float(np.clip(self.cost_total / STEPS_PER_WEEK, 0.0, 1.0))

        day, hour = self._current_day_hour()
        hour_sin = (np.sin(2 * np.pi * hour / 24) + 1) / 2
        hour_cos = (np.cos(2 * np.pi * hour / 24) + 1) / 2
        day_norm = day / 6.0

        hint = [self.hint_active, self.hint_magnitude, self.hint_time_to_event]

        return np.concatenate([
            avg_cpu_hist, max_cpu_hist, avg_mem_hist, max_mem_hist,
            [queue_depth_ratio], [queue_growth], [sla_pressure], [min_slack],
            [active_vms_norm], [cost_norm], [hour_sin], [hour_cos], [day_norm],
            hint,
        ]).astype(np.float32)

    # ----------------------------------------------------------- the step
    def step(self, action):
        # 1. apply action: [-1,1] -> up to +/-5 VMs
        delta = int(round(float(action[0]) * 5))
        self.active_vms = int(np.clip(self.active_vms + delta, MIN_PODS, MAX_PODS))

        # 2. generate jobs
        self.queue.extend(self._generate_jobs())

        # 3. assign to VMs (CPU + memory)
        jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem = self._assign_jobs()

        # 4. deadlines -> breaches
        breaches = self._check_deadlines()
        self.total_breaches += breaches

        # 5. cost & utilisation
        cost = self.active_vms / MAX_PODS
        self.cost_total += cost
        capacity = self.active_vms * JOBS_PER_STEP_PER_VM
        utilisation = jobs_processed / capacity if capacity > 0 else 0.0

        # normalise breaches into a 0-1 rate so it can't dwarf cost
        jobs_due_this_step = jobs_processed + breaches   # rough denominator
        breach_rate = breaches / jobs_due_this_step if jobs_due_this_step > 0 else 0.0

        # 6. reward
        reward = (- LAMBDA_COST * cost
                  - LAMBDA_SLA * breach_rate
                  + LAMBDA_UTIL * utilisation)

        # 7. history (real memory values)
        self.history.append([avg_cpu, max_cpu, avg_mem, max_mem])
        self.history.pop(0)

        # log decision for Phase 2 RAG
        day, hour = self._current_day_hour()
        self.decision_log.append({
            'step': self.step_count, 'day': day, 'hour': hour,
            'active_vms': self.active_vms,
            'avg_cpu': round(avg_cpu, 3), 'avg_mem': round(avg_mem, 3),
            'queue': len(self.queue), 'breaches': breaches,
            'reward': round(reward, 3),
        })

        self.prev_queue_len = len(self.queue)
        self.step_count += 1

        obs = self._build_state()
        done = self.step_count >= STEPS_PER_WEEK
        info = {'cost': cost, 'breaches': breaches, 'utilisation': utilisation,
                'active_vms': self.active_vms, 'queue': len(self.queue),
                'avg_cpu': avg_cpu, 'avg_mem': avg_mem}
        return obs, reward, done, False, info

print("CloudClusterEnv defined.")

CloudClusterEnv defined.


## 3. Sanity check — random agent

Runs one full week with random actions to confirm the environment works
end-to-end. A random agent should rack up many SLA breaches (it scales
badly) — that's expected and correct. The trained PPO agent will bring
this down dramatically.

In [3]:
env = CloudClusterEnv()
obs, _ = env.reset()
print("Initial state shape:", obs.shape, "(should be 32)")
print("Running one week with random actions...\n")

total_reward = total_cost = total_breaches = 0
vm_counts = []
for t in range(STEPS_PER_WEEK):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    total_cost += info['cost']
    total_breaches += info['breaches']
    vm_counts.append(info['active_vms'])
    if done:
        break

print(f"Steps run:          {t+1}")
print(f"Total reward:       {total_reward:.1f}")
print(f"Total cost:         {total_cost:.1f}")
print(f"Total SLA breaches: {total_breaches}")
print(f"Avg VMs used:       {np.mean(vm_counts):.1f}")
print(f"Min/Max VMs:        {min(vm_counts)} / {max(vm_counts)}")
print(f"State is 32-dim:    {obs.shape == (32,)}")

Initial state shape: (32,) (should be 32)
Running one week with random actions...

Steps run:          672
Total reward:       -22407.5
Total cost:         384.8
Total SLA breaches: 22325
Avg VMs used:       11.5
Min/Max VMs:        2 / 20
State is 32-dim:    True
